In [1]:
import os
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE = "https://www.federalreserve.gov"
INDEX_URL = f"{BASE}/monetarypolicy/publications/mpr_default.htm"
TXT_DIR = "txt"
CSV_DIR = "out/csv"

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; mpr-scraper/1.0; research use)"}

MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12,
}

# Modern full-report PDFs: .../files/YYYYMMDD_mprfullreport.pdf
_MODERN = re.compile(r"(\d{8})_mpr\w*\.pdf", re.I)
# Legacy full-report PDFs: .../boarddocs/hh/YYYY/<month>/FullReport.pdf
_LEGACY = re.compile(r"/hh/(\d{4})/([a-z]+)/full\w*report\.pdf", re.I)

def get(url, tries=3):
    """GET with a real User-Agent, a timeout, and a couple of retries."""
    for i in range(tries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=120)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if i == tries - 1:
                raise
            time.sleep(2 * (i + 1))

def find_reports(html):
    """Parse the MPR index page and return a sorted list of
    (year, month, day, pdf_url) tuples, one per full report.
    `day` is None for legacy reports whose URL doesn't encode it."""
    soup = BeautifulSoup(html, "html.parser")
    found = {}
    for a in soup.find_all("a", href=True):
        href = urljoin(BASE, a["href"])
        m = _MODERN.search(href)
        if m:
            d = m.group(1)
            year, month, day = int(d[:4]), int(d[4:6]), int(d[6:8])
            found[(year, month)] = (year, month, day, href)
            continue
        m = _LEGACY.search(href)
        if m:
            year = int(m.group(1))
            month = MONTHS.get(m.group(2).lower())
            if month:
                found.setdefault((year, month), (year, month, None, href))
    return sorted(found.values())

def get_index():
    """Fetch and parse the index page in one step."""
    return find_reports(get(INDEX_URL).text)

def get_txt_file(year, month):
    return f"{TXT_DIR}/{year}/{year}-{month:02d}-mpr.txt"

def get_txt_string(year, month):
    with open(get_txt_file(year, month), "r") as f:
        return f.read()

In [2]:
import os
import re
import time
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE = "https://www.federalreserve.gov"
INDEX_URL = f"{BASE}/monetarypolicy/publications/mpr_default.htm"
TXT_DIR = "txt"
CSV_DIR = "out/csv"

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; mpr-scraper/1.0; research use)"}

MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12,
}

# Modern full-report PDFs: .../files/YYYYMMDD_mprfullreport.pdf
_MODERN = re.compile(r"(\d{8})_mpr\w*\.pdf", re.I)
# Legacy full-report PDFs: .../boarddocs/hh/YYYY/<month>/FullReport.pdf
_LEGACY = re.compile(r"/hh/(\d{4})/([a-z]+)/full\w*report\.pdf", re.I)

def get(url, tries=3):
    """GET with a real User-Agent, a timeout, and a couple of retries."""
    for i in range(tries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=120)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if i == tries - 1:
                raise
            time.sleep(2 * (i + 1))

def find_reports(html):
    """Parse the MPR index page and return a sorted list of
    (year, month, day, pdf_url) tuples, one per full report.
    `day` is None for legacy reports whose URL doesn't encode it."""
    soup = BeautifulSoup(html, "html.parser")
    found = {}
    for a in soup.find_all("a", href=True):
        href = urljoin(BASE, a["href"])
        m = _MODERN.search(href)
        if m:
            d = m.group(1)
            year, month, day = int(d[:4]), int(d[4:6]), int(d[6:8])
            found[(year, month)] = (year, month, day, href)
            continue
        m = _LEGACY.search(href)
        if m:
            year = int(m.group(1))
            month = MONTHS.get(m.group(2).lower())
            if month:
                found.setdefault((year, month), (year, month, None, href))
    return sorted(found.values())

def get_index():
    """Fetch and parse the index page in one step."""
    return find_reports(get(INDEX_URL).text)

def get_txt_file(year, month):
    return f"{TXT_DIR}/{year}/{year}-{month:02d}-mpr.txt"

def get_txt_string(year, month):
    with open(get_txt_file(year, month), "r") as f:
        return f.read()